# On the origins of Claudish: RL and the origins of model idiolect

**What reward optimization does to the one coordinate its verifier cannot see.**

A verifier that accepts any grammatical answer is *exactly invariant* to which of several
interchangeable words the model used to say a thing. Pretraining is not: its whole objective
is calibration to the corpus's surface forms. So when reward optimization takes over, nothing
is holding word choice in place any more. This notebook measures what happens on that free
coordinate while the verifier's own number goes up.

It is a self-contained, runnable version of
[`experiments/rhm/rl_dimensionality/idiolect/`](https://github.com/jagilley/abstraction/tree/main/experiments/rhm/rl_dimensionality/idiolect)
in Jasper Gilley's research monorepo, and it does three things:

1. **Explains the setup** — a hierarchical grammar with a dial for "how many ways to say the
   same thing", a task, two verifiers, and an *exact* readout of which synonym the model chose
   at every node it generated. Not an estimate: ground truth, recovered by inverting the grammar.
2. **Runs a scaled-down replica live** (~15 min on a free Colab T4): pretrain one checkpoint, then
   run expert iteration from it twice with different seeds, once more with the *same* seed as a
   noise floor, and once under the other verifier — then read out how far each run's dialect moved
   and in what direction.
3. **Shows the full-scale results** — every checkpoint of the Modal sweep, embedded in the notebook:
   five settings of the dial, six mechanisms, both verifiers, two seeds, and all six expert-iteration
   rounds — with the interpretation from the writeup inline.

> **Runtime → Change runtime type → GPU** before running. Everything works on CPU too, just slowly;
> set `RUN_LIVE = False` in the configuration cell to skip the live training and read only the
> embedded results.

The substrate is the Random Hierarchy Model of Cagnetta & Wyart (*Phys. Rev. X*, 2024). This notebook
follows the deck `papers/rl_idiolect_slides.tex`; its parent experiment ("RL is not enough when there
are many right answers") has its own notebook, and this one recaps what it needs from it.

In [ ]:
#@title Setup { display-mode: "form" }
import os, math, json, time, gzip, base64
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Wedge

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
SMOKE = os.environ.get("IDIOLECT_SMOKE") == "1"   # tiny sizes for CI-style testing only
print(f"torch {torch.__version__} | device: {device}" + ("  [SMOKE]" if SMOKE else ""))

# --- palette (validated categorical set; gray is the neutral reference, not a series) ---
PAL = dict(seedA="#eb6834", seedB="#4a3aa7", exact="#2a78d6", anchor="#1baf7a",
           collapse="#e34948", gray="#9a9891", light="#e6e4df", ink="#0b0b0b", muted="#52514e")
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": PAL["light"], "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.edgecolor": PAL["muted"], "axes.labelcolor": PAL["ink"], "xtick.color": PAL["muted"],
    "ytick.color": PAL["muted"], "font.size": 10, "legend.frameon": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
})
MS_FULL = [1, 2, 3, 4, 6]
ALPHABET = "abcdefgh"     # the eight tokens, given letters so words read as words

In [ ]:
#@title Embedded full-scale results (every checkpoint of the Modal sweep, sampled decoding) { display-mode: "form" }
FULL_RESULTS_B64 = """
...
"""
FULL = json.loads(gzip.decompress(base64.b64decode(FULL_RESULTS_B64.strip())))
del FULL_RESULTS_B64

FCFG = FULL["config"]
def fs(m, label):
    """One condition's full-scale record at multiplicity m."""
    return FULL["by_m"][str(m)]["conditions"][label]

print(f"full-scale sweep: v={FCFG['v']}, s={FCFG['s']}, L={FCFG['L']}, "
      f"prefix={FCFG['prefix_len']}, {FCFG['n_eval']} sampled generations per checkpoint, "
      f"temperature {FCFG['temperature']}")
print(f"  m values: {sorted(int(k) for k in FULL['by_m'])}")
print(f"  {len(FULL['by_m']['6']['conditions'])} conditions per m, e.g. "
      f"{', '.join(list(FULL['by_m']['6']['conditions'])[:6])}, ...")

## 1. The question

The parent experiment put a pretrained transformer through three reward-optimization mechanisms on a
synthetic language, and varied one knob: **m**, how many interchangeable rules ("synonyms") can realize
each latent feature. Two verifiers graded the same task:

- **canonical-answer** — score the generation against *one* sampled ground-truth continuation. A synonym
  counts as wrong.
- **validity** — score how much of the generation parses as grammatical. Any grammatical continuation
  passes.

Two things came out of it that set up this notebook.

**The canonical-answer verifier empties out as m grows.** Its prefix-conditional headroom — how much
better than a prefix-blind guesser any policy could possibly do — is 0.096 at m=1 and ~0.0000 at m=6, by
exact computation. Deep hierarchies mix: by the time you are six levels down, the prefix tells you almost
nothing token-by-token about the suffix. At high synonymy there is nothing there to optimize toward.

**The validity verifier keeps its signal at every m** — and it is the one that survives into the
interesting regime. But it buys that signal by being *exactly invariant* to one coordinate: it does not
care which of the m synonyms you used. Every one of them parses.

So: **what does reward do to a coordinate it cannot see?**

### The observation this is a mechanism for

Recent models coin esoteric terms with private definitions — well-formed, evidently meaningful to the
model, hard for an outside reader to pin down. There was much less of this in the pretrain-and-instruct
era. The framework above suggests a mechanism worth measuring, not a conclusion:

> Pretraining's objective **is** calibration to the corpus's surface forms, so it is the only force pinning
> word choice to shared convention. A reward that cannot see word choice does not push toward idiosyncrasy.
> It **stops holding vocabulary in place**.

Whether anything then drifts, how far, and whether the direction is shared or arbitrary are empirical
questions. Three of them:

1. **Does anything drift at all?**
2. **How much — and does it come packaged with improvement, or with damage?**
3. **Is the direction shared between independent runs, or private to each?**

## 2. The grammar, and a dial for "how many ways to say the same thing"

The Random Hierarchy Model is a probabilistic context-free grammar with a fixed shape. There are `v = 8`
features. A sequence starts as one random root feature and is expanded `L = 6` times: at every step, each
feature picks one of its `m` rules uniformly at random and is replaced by that rule's `s = 2` children.
After six expansions you have `s^L = 64` leaf tokens.

`m` is the whole experiment's knob. It is the surface-to-latent degeneracy — how many different ways the
grammar has of saying the same thing — and it moves the semantic sample complexity `m^L` while holding
sequence length, tree shape, architecture and compute-per-rollout **exactly** fixed. (Varying `L` or `s`
would move complexity too, but would also change the horizon, which is a confound with a large literature
of its own.) At `m = 1` there are exactly 8 possible sequences in the language; at `m = 8` the grammar
emits the uniform distribution and there is no structure left. The sweep runs `m ∈ {1, 2, 3, 4, 6}`.

One construction detail is load-bearing here. The rules are sampled **collision-free**: across a whole
level, the `v·m` produced pairs are all distinct, so every legal pair maps back to exactly one
`(feature, rule)`. That makes the parse unique and, as the next section uses, makes *which synonym was
used* exactly recoverable from the tokens alone.

In [ ]:
# ----------------------------------------------------------------------------
# The generative process. Ported verbatim from rhm/rhm_data.py, numpy only.
# ----------------------------------------------------------------------------

def generate_rules_invertible(v, s, L, m, seed=0):
    """Composition rules whose produced s-tuples are all distinct within a level.

    Returns L arrays of shape (v, m, s): rules[ell][f, r] is the s-tuple that rule
    r of feature f expands to. Level 0 is the top (root -> its children), level
    L-1 the bottom (features -> leaf tokens). Collision-free sampling makes the
    bottom-up parse unique, which is what section 3 needs.
    """
    rng = np.random.default_rng(seed)
    n_codes = v ** s
    if v * m > n_codes:
        raise ValueError(f"collision-free rules require v*m <= v**s (got {v*m} > {n_codes})")
    powers = v ** np.arange(s)
    rules = []
    for _ in range(L):
        codes = rng.choice(n_codes, size=v * m, replace=False).reshape(v, m)
        layer = np.empty((v, m, s), dtype=np.int64)
        for i in range(s):
            layer[:, :, i] = (codes // (v ** i)) % v
        assert np.array_equal((layer * powers).sum(axis=2).reshape(-1), codes.reshape(-1))
        rules.append(layer)
    return rules


def generate_sequences_batched(rules, n_sequences, seed=0, batch_size=10000):
    """Sample n_sequences from the grammar: random root, L uniform expansions."""
    rng = np.random.default_rng(seed)
    L = len(rules)
    v, m, s = rules[0].shape
    out = np.empty((n_sequences, s ** L), dtype=np.int64)
    for start in range(0, n_sequences, batch_size):
        end = min(start + batch_size, n_sequences)
        bs = end - start
        cur = rng.integers(0, v, size=(bs, 1))
        for ell in range(L):
            n_feat = cur.shape[1]
            choices = rng.integers(0, m, size=(bs, n_feat))
            nxt = np.empty((bs, n_feat * s), dtype=np.int64)
            for j in range(n_feat):
                nxt[:, j * s:(j + 1) * s] = rules[ell][cur[:, j], choices[:, j]]
            cur = nxt
        out[start:end] = cur
    return out


def generate_with_traces(rules, n_sequences, seed=999):
    """Same sampler, but also returns the ground-truth feature and rule at every node.

    Used only to check that the recovery in section 3 is exact -- the recovery
    itself never sees these.
    """
    rng = np.random.default_rng(seed)
    L = len(rules)
    v, m, s = rules[0].shape
    level_features, level_rules = [], []
    cur = rng.integers(0, v, size=(n_sequences, 1))
    for ell in range(L):
        level_features.append(cur.copy())
        n_nodes = cur.shape[1]
        rc = rng.integers(0, m, size=(n_sequences, n_nodes))
        level_rules.append(rc.copy())
        nxt = np.empty((n_sequences, n_nodes * s), dtype=np.int64)
        for j in range(n_nodes):
            nxt[:, j * s:(j + 1) * s] = rules[ell][cur[:, j], rc[:, j]]
        cur = nxt
    return cur, level_features, level_rules


def possible_set_parse(leaves, rules, chunk=2048):
    """Exact bottom-up parse: carries the SET of features that could produce each node.

    Returns {"valid": (B,) bool}. Used as the independent reference the recovery
    in section 3 is checked against.
    """
    v, m, s = rules[0].shape
    L = len(rules)
    leaves = np.asarray(leaves, dtype=np.int64)
    B = leaves.shape[0]
    valid = np.zeros(B, dtype=bool)
    for lo in range(0, B, chunk):
        hi = min(lo + chunk, B)
        cur = np.zeros((hi - lo, leaves.shape[1], v), dtype=bool)
        np.put_along_axis(cur, leaves[lo:hi][:, :, None], True, axis=2)
        alive = np.ones(hi - lo, dtype=bool)
        for ell in range(L - 1, -1, -1):
            b, n, _ = cur.shape
            children = cur.reshape(b, n // s, s, v)
            ok = np.ones((b, n // s, v, m), dtype=bool)
            for i in range(s):
                ok &= children[:, :, i, :][:, :, rules[ell][:, :, i]]
            cur = ok.any(axis=3)
            alive &= cur.any(axis=2).all(axis=1)
        valid[lo:hi] = alive
    return {"valid": valid}


V, S, L_DEPTH, RULE_SEED = 8, 2, 6, 0
SEQ_LEN, PREFIX_LEN = S ** L_DEPTH, (S ** L_DEPTH) // 2
print(f"v={V}  s={S}  L={L_DEPTH}  ->  sequences of {SEQ_LEN} tokens, prefix = {PREFIX_LEN}")

In [ ]:
# The m synonyms of a bottom-level feature are literally m words for one thing:
# the bottom rules expand a feature straight into s leaf tokens.
def words_of(rules, feature):
    bottom = rules[-1]                                 # (v, m, s) -> leaf tokens
    return ["".join(ALPHABET[t] for t in bottom[feature, r]) for r in range(bottom.shape[1])]

demo_rules = generate_rules_invertible(V, S, L_DEPTH, 3, seed=RULE_SEED)
print("m = 3, the bottom level of the grammar — every feature has three words for itself:\n")
for f in range(V):
    print(f"  feature {f}:  " + "   ".join(words_of(demo_rules, f)))

# and the corpus uses them evenly, by construction
corpus = generate_sequences_batched(demo_rules, 4000, seed=1)
print(f"\na corpus sample:  {''.join(ALPHABET[t] for t in corpus[0])}")
print(f"                  {''.join(ALPHABET[t] for t in corpus[1])}")

## 3. The coordinate the verifier cannot see, and how to read it exactly

Because the rules are collision-free, a bottom-up fold recovers the `(feature, rule)` pair at **every**
internal node of a sequence — the model's synonym choices are ground truth, not an estimate. Statistics are
taken only over nodes whose *entire subtree* lies inside the model-generated suffix, so the choice being
counted was the model's and not the prompt's.

Two numbers come out of the recovered counts.

**Drift**, in bits, per level and node-weighted overall:

$$\mathrm{drift} \;=\; \log_2 m \;-\; H(\text{synonym} \mid \text{feature})$$

The grammar picks synonyms uniformly, so **0 means the model has the corpus's habits** and $\log_2 m$ means
it has committed to one synonym for everything. This is the KL from the model's synonym distribution to the
corpus's, conditioned on what is being said.

**Distance** between two models, for asking whether two runs drifted the *same way*: the Jensen–Shannon
divergence between their feature-conditioned synonym distributions, weighted by feature frequency. Its square
root is a metric, so three models' pairwise distances form a real triangle and the angle at their common
ancestor follows from the law of cosines — which is how "did these two runs drift in the same direction?"
becomes a number.

In [ ]:
# ----------------------------------------------------------------------------
# Exact synonym recovery and the two statistics.
# Ported verbatim from rhm/rl_dimensionality/idiolect/idiolect_drift.py.
# ----------------------------------------------------------------------------

def build_rule_inverse(rules):
    """Per-level tuple-code -> (feature, rule) tables. Requires collision-free rules."""
    v, m, s = rules[0].shape
    powers = v ** np.arange(s)
    tables = []
    for layer in rules:
        inv_feat = np.full(v ** s, -1, dtype=np.int64)
        inv_rule = np.full(v ** s, -1, dtype=np.int64)
        codes = (layer * powers).sum(axis=2)                    # (v, m)
        if len(np.unique(codes)) != codes.size:
            raise ValueError("rules are not collision-free; recovery is ambiguous")
        inv_feat[codes.reshape(-1)] = np.repeat(np.arange(v), m)
        inv_rule[codes.reshape(-1)] = np.tile(np.arange(m), v)
        tables.append((inv_feat, inv_rule))
    return tables


def recover_rule_usage(seqs, rules, tables=None):
    """Bottom-up exact recovery of (feature, rule) at every internal node.

    Returns a list of L dicts, index ell holding (B, s^ell) arrays: `feat`,
    `rule` (-1 where the node does not parse) and `valid`. Invalidity propagates
    upward exactly as in possible_set_parse.
    """
    v, m, s = rules[0].shape
    L = len(rules)
    tables = build_rule_inverse(rules) if tables is None else tables
    powers = v ** np.arange(s)
    cur = np.asarray(seqs, dtype=np.int64)
    B = cur.shape[0]
    valid = np.ones_like(cur, dtype=bool)
    out = [None] * L
    for ell in range(L - 1, -1, -1):
        n2 = cur.shape[1] // s
        child = cur.reshape(B, n2, s)
        child_valid = valid.reshape(B, n2, s).all(axis=2)
        code = (child * powers).sum(axis=2)
        inv_feat, inv_rule = tables[ell]
        feat, rule = inv_feat[code], inv_rule[code]
        node_valid = child_valid & (feat >= 0)
        out[ell] = {"feat": np.where(node_valid, feat, -1),
                    "rule": np.where(node_valid, rule, -1),
                    "valid": node_valid}
        cur = np.where(node_valid, feat, 0)
        valid = node_valid
    return out


def free_node_slice(ell, prefix_len, seq_len, s):
    """Level-ell nodes whose whole subtree lies inside the generated suffix."""
    n_nodes = s ** ell
    span = seq_len // n_nodes
    return -(-prefix_len // span), n_nodes                      # (first_free, n_nodes)


def _entropy_bits(counts, axis=-1):
    p = counts / np.clip(counts.sum(axis=axis, keepdims=True), 1e-30, None)
    with np.errstate(divide="ignore", invalid="ignore"):
        logp = np.where(p > 0, np.log2(p), 0.0)
    return -(p * logp).sum(axis=axis)


def idiolect_stats(seqs, rules, prefix_len, tables=None):
    """Per-level synonym-choice statistics over model-chosen nodes.

    Per level: n_nodes, n_valid, valid_frac (grammaticality), H_marg, H_cond,
    kl_cond = log2(m) - H_cond (the drift), and the (v, m) joint histogram.
    Plus "overall", the node-weighted mean across levels.
    """
    v, m, s = rules[0].shape
    L = len(rules)
    seq_len = s ** L
    usage = recover_rule_usage(seqs, rules, tables)
    per_level = {}
    for ell in range(L):
        first_free, n_nodes = free_node_slice(ell, prefix_len, seq_len, s)
        if first_free >= n_nodes:
            continue
        feat = usage[ell]["feat"][:, first_free:]
        rule = usage[ell]["rule"][:, first_free:]
        ok = usage[ell]["valid"][:, first_free:]
        n_total, n_valid = int(ok.size), int(ok.sum())
        entry = {"n_nodes": n_total, "n_valid": n_valid,
                 "valid_frac": n_valid / max(n_total, 1)}
        if n_valid == 0 or m == 1:
            entry.update({"H_marg": 0.0, "H_cond": 0.0, "kl_marg": 0.0, "kl_cond": 0.0,
                          "joint_hist": [[0.0] * m for _ in range(v)]})
            per_level[f"L{ell}"] = entry
            continue
        joint = np.zeros((v, m), dtype=np.float64)
        np.add.at(joint, (feat[ok], rule[ok]), 1.0)
        marg = joint.sum(axis=1)
        H_marg = float(_entropy_bits(joint.sum(axis=0)))
        H_cond = float((marg * _entropy_bits(joint, axis=1)).sum() / max(marg.sum(), 1e-30))
        entry.update({"H_marg": H_marg, "H_cond": H_cond,
                      "kl_marg": float(np.log2(m) - H_marg),
                      "kl_cond": float(np.log2(m) - H_cond),
                      "joint_hist": (joint / max(joint.sum(), 1e-30)).tolist()})
        per_level[f"L{ell}"] = entry
    tot = sum(e["n_nodes"] for e in per_level.values()) or 1
    totv = sum(e["n_valid"] for e in per_level.values()) or 1
    overall = {"valid_frac": sum(e["valid_frac"] * e["n_nodes"] for e in per_level.values()) / tot}
    for k in ("H_marg", "H_cond", "kl_marg", "kl_cond"):
        overall[k] = sum(e[k] * e["n_valid"] for e in per_level.values()) / totv
    per_level["overall"] = overall
    return per_level


def js_divergence_bits(p, q):
    """Jensen-Shannon divergence in bits between two (v, m) joint histograms.

    Compares the two models' *conditional* rule distributions, weighted by the
    average feature frequency, so it measures disagreement about synonym choice
    rather than disagreement about which features get produced.
    """
    p, q = np.asarray(p, dtype=np.float64), np.asarray(q, dtype=np.float64)
    if p.sum() <= 0 or q.sum() <= 0:
        return 0.0
    p, q = p / p.sum(), q / q.sum()
    wp, wq = p.sum(axis=1), q.sum(axis=1)
    w = 0.5 * (wp + wq)
    cp = p / np.clip(wp[:, None], 1e-30, None)
    cq = q / np.clip(wq[:, None], 1e-30, None)
    mid = 0.5 * (cp + cq)

    def _kl(a, b):
        with np.errstate(divide="ignore", invalid="ignore"):
            t = np.where(a > 0, a * np.log2(np.clip(a, 1e-30, None) / np.clip(b, 1e-30, None)), 0.0)
        return t.sum(axis=1)

    return float((w * (0.5 * _kl(cp, mid) + 0.5 * _kl(cq, mid))).sum())


def dialect_distance(levels_a, levels_b):
    """sqrt of the node-weighted JS across levels: the metric the triangles use.

    `levels_*` is the per-level dict from idiolect_stats (or the embedded extract's
    "levels" block). Weighting by valid-node counts matches the full-scale sweep.
    """
    tot = acc = 0.0
    for k, a in levels_a.items():
        if not k.startswith("L") or k not in levels_b:
            continue
        b = levels_b[k]
        w = a["n_valid"] + b["n_valid"]
        acc += w * js_divergence_bits(a["joint_hist"], b["joint_hist"])
        tot += w
    return float(np.sqrt(acc / max(tot, 1e-30)))


def drift_angle(d_ab, d_ac, d_bc):
    """Angle at a between the drifts a->b and a->c, by the law of cosines."""
    if d_ab <= 0 or d_ac <= 0:
        return float("nan")
    cos = (d_ab ** 2 + d_ac ** 2 - d_bc ** 2) / (2 * d_ab * d_ac)
    return float(np.degrees(np.arccos(np.clip(cos, -1.0, 1.0))))

### Does the recovery actually work? Four checks, live

Before trusting a number, check the instrument. These are the self-tests from the experiment
(`idiolect_drift::self_test`), run here on CPU in about a minute:

1. **Recovery is exact.** Sample from the grammar while recording the true feature and rule at every node,
   then recover them from the leaf tokens alone. They must match at every level and every `m`.
2. **Validity agrees with an independent parser.** Corrupt some sequences and check the recovered
   validity flag against the set-carrying CYK parse.
3. **The drift metric is calibrated at both ends.** True corpus samples must read ≈ 0 bits; a speaker
   forced to always use rule 0 must read exactly $\log_2 m$.
4. **The distance is calibrated at both ends.** Identical distributions → 0; disjoint → 1 bit.

In [ ]:
def generate_forced(rules, n, rule_choice=0, seed=0):
    """Grammar sampling with the rule index pinned: a maximally idiosyncratic speaker."""
    rng = np.random.default_rng(seed)
    v, m, s = rules[0].shape
    cur = rng.integers(0, v, size=(n, 1))
    for ell in range(len(rules)):
        n_nodes = cur.shape[1]
        nxt = np.empty((n, n_nodes * s), dtype=np.int64)
        for j in range(n_nodes):
            nxt[:, j * s:(j + 1) * s] = rules[ell][cur[:, j], rule_choice]
        cur = nxt
    return cur

n_trace, n_cal = (60, 200) if SMOKE else (500, 4000)
ok = True

print("1. recovery vs ground-truth generation traces")
for m in (1, 2, 3, 4, 6):
    rules = generate_rules_invertible(V, S, L_DEPTH, m, seed=RULE_SEED)
    seqs, lf, lr = generate_with_traces(rules, n_trace, seed=7)
    usage = recover_rule_usage(seqs, rules)
    good = all(np.array_equal(usage[e]["feat"], lf[e]) and
               np.array_equal(usage[e]["rule"], lr[e]) and usage[e]["valid"].all()
               for e in range(L_DEPTH))
    ok &= good
    print(f"   m={m}: all {L_DEPTH} levels recover feature and rule exactly -> {good}")

print("\n2. validity agrees with the set-carrying parse on corrupted sequences")
for m in (2, 4):
    rules = generate_rules_invertible(V, S, L_DEPTH, m, seed=RULE_SEED)
    seqs, _, _ = generate_with_traces(rules, max(80, n_trace), seed=11)
    rng = np.random.default_rng(3)
    corrupt = seqs.copy()
    k = corrupt.shape[0] // 2
    corrupt[:k, rng.integers(0, SEQ_LEN, size=k)] = rng.integers(0, V, size=k)
    agree = bool(np.array_equal(possible_set_parse(corrupt, rules)["valid"],
                                recover_rule_usage(corrupt, rules)[0]["valid"][:, 0]))
    ok &= agree
    print(f"   m={m}: matches -> {agree}")

print("\n3. drift metric calibration (corpus ~ 0 bits; always-rule-0 = log2 m exactly)")
for m in (2, 4, 6):
    rules = generate_rules_invertible(V, S, L_DEPTH, m, seed=RULE_SEED)
    seqs, _, _ = generate_with_traces(rules, n_cal, seed=13)
    st = idiolect_stats(seqs, rules, PREFIX_LEN)["overall"]["kl_cond"]
    sf = idiolect_stats(generate_forced(rules, n_cal, 0, seed=13), rules, PREFIX_LEN)["overall"]["kl_cond"]
    good = st < 0.05 and abs(sf - np.log2(m)) < 1e-6
    ok &= good
    print(f"   m={m}: corpus {st:.4f} bits (floor), forced {sf:.4f} (max {np.log2(m):.4f}) -> {good}")

print("\n4. distance calibration")
p, q = [[0.5, 0.0], [0.5, 0.0]], [[0.0, 0.5], [0.0, 0.5]]
good = abs(js_divergence_bits(p, p)) < 1e-12 and abs(js_divergence_bits(p, q) - 1.0) < 1e-12
ok &= good
print(f"   identical {js_divergence_bits(p, p):.6f}, disjoint {js_divergence_bits(p, q):.6f} -> {good}")

print("\n" + ("ALL CHECKS PASSED" if ok else "SOME CHECKS FAILED"))

### The floor the drift has to beat

Entropy estimated from a finite sample is biased downward, so `kl_cond = log2(m) − H(rule | feature)` is
biased *up*: even a perfect speaker of the corpus's dialect reads slightly above zero. The reference is
therefore not 0 but the same statistic computed on **true grammar samples at matched node counts** — and
this one is cheap enough to run live, at the full sweep's sample size, for every `m`.

The cell below computes it and checks it against the number the full-scale sweep recorded. They are
independent samples of the same quantity, so they should agree to within sampling noise, not exactly.

In [ ]:
n_floor = 300 if SMOKE else FCFG["n_eval"]
print(f"corpus floor for the drift metric, {n_floor} samples per m (full sweep used {FCFG['n_eval']}):\n")
print(f"  {'m':>3}  {'live floor':>12}  {'full-scale floor':>18}  {'max = log2 m':>14}")
DGP_FLOOR = {}
for m in MS_FULL:
    rules = generate_rules_invertible(V, S, L_DEPTH, m, seed=RULE_SEED)
    live = idiolect_stats(generate_sequences_batched(rules, n_floor, seed=FCFG["eval_seed"]),
                          rules, PREFIX_LEN)["overall"]["kl_cond"]
    DGP_FLOOR[m] = live
    print(f"  {m:>3}  {live:>12.4f}  {fs(m, 'dgp')['overall']['kl_cond']:>18.4f}  {np.log2(m):>14.4f}")
print("\nTwo independent samples of the same quantity, so agreeing to the printed precision is the\n"
      "check. Everything reported later is read against this floor: the pretrained model sits\n"
      "within ~5x of it, and expert iteration ends up as much as 70x above it.")

## 4. The task, the two verifiers, and expert iteration

**Task.** The first 32 tokens of a 64-token sequence are given; generate the remaining 32 autoregressively
at temperature 1.

**Two verifiers**, scoring the same generation:

- **exact** (canonical-answer) — the fraction of suffix tokens matching *the* sampled ground-truth suffix.
  Word choice is part of the grade, and getting a synonym "wrong" costs you.
- **parse** (validity) — the fraction of nodes that parse, averaged over hierarchy levels. Any grammatical
  continuation scores. **Exactly invariant to synonym choice**: this is the verifier with the blind spot.

**Mechanism.** The main arm is **expert iteration**: sample N continuations per prompt, keep the one the
verifier likes best, fine-tune on the winners, repeat. It is the noiseless, perfectly-credit-assigned limit
of the sample-reweighting class — every policy-gradient method is a noisy approximation of *raise the
probability of your own high-reward samples* — with no other moving parts. The full run gave it a rollout
budget matched to the REINFORCE arms: 6 rounds × 4000 prompts × best-of-16 = 384K rollouts.

**The arms.** Everything below starts from one and the same pretrained checkpoint:

| arm | what it is |
|---|---|
| **EI, seed 42** | expert iteration under the validity verifier — the main arm |
| **EI, seed 43** | identical checkpoint, verifier and budget; only the seed differs — the privacy test |
| **same-seed rerun** | identical config *and* seed; float nondeterminism only — the noise floor |
| **EI, exact verifier** | the same loop under the canonical-answer verifier — the blind-spot control |
| **REINFORCE** | vanilla policy gradient: what entropy collapse looks like, for contrast |

In [ ]:
# ----------------------------------------------------------------------------
# The model (rhm/model.py, trimmed) and the two verifiers (rl_dim_ablation.py).
# ----------------------------------------------------------------------------

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf")).softmax(dim=-1)
        return self.c_proj((att @ v).transpose(1, 2).contiguous().view(B, T, C))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.ln_1, self.ln_2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        return x + self.mlp(self.ln_2(x))


class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd, quiet=False):
        super().__init__()
        self.block_size = block_size
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(vocab_size, n_embd), wpe=nn.Embedding(block_size, n_embd),
            h=nn.ModuleList([Block(n_embd, n_head, block_size) for _ in range(n_layer)]),
            ln_f=nn.LayerNorm(n_embd)))
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.apply(self._init_weights)
        if not quiet:
            print(f"GPT: {sum(p.numel() for p in self.parameters()) / 1e6:.2f}M parameters")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        x = self.transformer.wte(idx) + self.transformer.wpe(torch.arange(T, device=idx.device))
        for block in self.transformer.h:
            x = block(x)
        logits = self.lm_head(self.transformer.ln_f(x))
        loss = None if targets is None else F.cross_entropy(
            logits.view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss


def parse_valid_fractions_torch(seqs, rules_t):
    """Per-level fraction of nodes with a non-empty possible-set: the validity verifier.

    Column k is the fraction of nodes still parsing after fold k+1 (k=0 just above
    the leaves, k=L-1 the root). Torch port of possible_set_parse.
    """
    B = seqs.shape[0]
    v, m, s = rules_t[0].shape
    cur = F.one_hot(seqs, v).bool()
    fracs = []
    for ell in range(len(rules_t) - 1, -1, -1):
        n2 = cur.shape[1] // s
        children = cur.view(B, n2, s, v)
        ok = torch.ones(B, n2, v, m, dtype=torch.bool, device=seqs.device)
        for i in range(s):
            idx = rules_t[ell][:, :, i].reshape(-1)
            ok &= children[:, :, i, :][:, :, idx].view(B, n2, v, m)
        cur = ok.any(dim=3)
        fracs.append(cur.any(dim=2).float().mean(dim=1))
    return torch.stack(fracs, dim=1)                            # (B, L)


@torch.no_grad()
def generate_suffix(model, prefix, suffix_len, temperature=1.0, greedy=False):
    x, outs = prefix, []
    for _ in range(suffix_len):
        nl = model(x)[0][:, -1, :]
        nt = nl.argmax(dim=-1, keepdim=True) if greedy else \
            torch.multinomial(F.softmax(nl / temperature, dim=-1), num_samples=1)
        outs.append(nt)
        x = torch.cat([x, nt], dim=1)
    return torch.cat(outs, dim=1)


def compute_reward(prefix, generated, true_suffix, verifier, rules_t):
    if verifier == "exact":                       # canonical-answer: one right continuation
        return (generated == true_suffix).float().mean(dim=1)
    if verifier == "parse":                       # validity: any grammatical continuation
        return parse_valid_fractions_torch(torch.cat([prefix, generated], dim=1), rules_t).mean(dim=1)
    raise ValueError(verifier)

In [ ]:
# ----------------------------------------------------------------------------
# Pretraining, expert iteration, REINFORCE — the loops from rl_dim_ablation.py
# with the Modal plumbing removed.
# ----------------------------------------------------------------------------

def pretrain_to_plateau(rules_ctx, cfg):
    """Sequence-aligned next-token prediction until aligned val loss stops improving.

    Pretrain-to-plateau rather than to a step budget: the claims are about what
    reward does to a *given* basis, and a fixed budget would confound "reward made
    it worse" with "pretraining got less far".
    """
    torch.manual_seed(cfg["seed"])
    model = GPT(V, SEQ_LEN, cfg["n_layer"], cfg["n_head"], cfg["n_embd"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.01)
    gen = torch.Generator().manual_seed(cfg["seed"])
    train_t, val_t = rules_ctx["train_t"], rules_ctx["val_t"]

    def val_loss():
        model.eval()
        with torch.no_grad():
            ls = [model(b[:, :-1].contiguous(), b[:, 1:].contiguous())[1].item()
                  for b in val_t.split(cfg["batch"])]
        return float(np.mean(ls))

    best, best_state, stale, curve, step = float("inf"), None, 0, [], 0
    t0 = time.time()
    while step < cfg["pre_max"]:
        model.train()
        idx = torch.randint(train_t.shape[0], (cfg["batch"],), generator=gen)
        b = train_t[idx].to(device)
        _, loss = model(b[:, :-1].contiguous(), b[:, 1:].contiguous())
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        step += 1
        if step % cfg["pre_eval_every"] == 0:
            vl = val_loss()
            curve.append((step, vl))
            if vl < best - cfg["min_delta"]:
                best, stale = vl, 0
                best_state = {k: p.detach().cpu().clone() for k, p in model.state_dict().items()}
            else:
                stale += 1
            if step >= cfg["pre_min"] and stale >= cfg["patience"]:
                break
    if best_state is None:
        best_state = {k: p.detach().cpu().clone() for k, p in model.state_dict().items()}
    print(f"  pretrained: {step} steps, val {best:.4f} (uniform = {np.log(V):.4f} nats), "
          f"{time.time() - t0:.0f}s")
    del model, opt
    return best_state, curve


def run_ei(start_state, verifier, seed, rules_ctx, cfg, measure, label):
    """Expert iteration: best-of-N under the verifier, then SFT on the winners.

    `measure` is called after every round with the current model; its return value
    is appended to the trajectory. Nothing here looks at synonym choice — the loop
    is exactly the parent experiment's, and the measurement is a bystander.
    """
    torch.manual_seed(seed)
    model = GPT(V, SEQ_LEN, cfg["n_layer"], cfg["n_head"], cfg["n_embd"], quiet=True).to(device)
    model.load_state_dict(start_state)
    gen = torch.Generator().manual_seed(seed + 200)
    rl_t, rules_t = rules_ctx["rl_t"], rules_ctx["rules_t"]
    suffix_len = SEQ_LEN - PREFIX_LEN
    per_pass = max(1, 1024 // cfg["ei_n"])
    traj = [measure(model, 0)]
    t0 = time.time()
    for rnd in range(cfg["ei_rounds"]):
        model.eval()
        win_seqs, r_sum, r_cnt = [], 0.0, 0
        remaining = cfg["ei_prompts"]
        while remaining > 0:
            k = min(per_pass, remaining); remaining -= k
            seqs = rl_t[torch.randint(rl_t.shape[0], (k,), generator=gen)].to(device)
            prefix, true_suffix = seqs[:, :PREFIX_LEN], seqs[:, PREFIX_LEN:]
            rep_p = prefix.repeat_interleave(cfg["ei_n"], dim=0)
            rep_t = true_suffix.repeat_interleave(cfg["ei_n"], dim=0)
            g = generate_suffix(model, rep_p, suffix_len, cfg["temperature"])
            r = compute_reward(rep_p, g, rep_t, verifier, rules_t).view(k, cfg["ei_n"])
            r_sum += float(r.sum()); r_cnt += k * cfg["ei_n"]
            best = r.argmax(dim=1)
            winners = g.view(k, cfg["ei_n"], suffix_len)[torch.arange(k, device=device), best]
            win_seqs.append(torch.cat([prefix, winners], dim=1).cpu())
        win = torch.cat(win_seqs)

        model.train()                                   # SFT on winners, suffix loss only
        opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.01)
        for _ in range(cfg["ei_sft_epochs"]):
            perm = torch.randperm(win.shape[0], generator=gen)
            for i in range(0, win.shape[0] - cfg["batch"] + 1, cfg["batch"]):
                b = win[perm[i:i + cfg["batch"]]].to(device)
                logits, _ = model(b[:, :-1].contiguous())
                ce = F.cross_entropy(logits.reshape(-1, V), b[:, 1:].reshape(-1),
                                     reduction="none").view(-1, SEQ_LEN - 1)
                loss = ce[:, PREFIX_LEN - 1:].mean()
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        del opt
        rec = measure(model, rnd + 1)
        rec["sample_reward"] = r_sum / max(1, r_cnt)
        traj.append(rec)
        print(f"    [{label}] round {rnd+1}: reward {rec['sample_reward']:.4f}  "
              f"drift {rec['kl_cond']:.4f} bits  root valid {rec['root_valid']:.3f}  "
              f"grammatical {rec['valid_frac']:.4f}")
    print(f"    [{label}] {time.time() - t0:.0f}s")
    state = {k: p.detach().cpu().clone() for k, p in model.state_dict().items()}
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return {"trajectory": traj, "state": state}


def run_reinforce(start_state, verifier, seed, rules_ctx, cfg, measure, label):
    """Vanilla REINFORCE with an EMA baseline — no anchor. The collapse contrast."""
    torch.manual_seed(seed)
    model = GPT(V, SEQ_LEN, cfg["n_layer"], cfg["n_head"], cfg["n_embd"], quiet=True).to(device)
    model.load_state_dict(start_state)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.01)
    gen = torch.Generator().manual_seed(seed + 100)
    rl_t, rules_t = rules_ctx["rl_t"], rules_ctx["rules_t"]
    suffix_len = SEQ_LEN - PREFIX_LEN
    baseline, traj = None, [measure(model, 0)]
    t0 = time.time()
    for st in range(cfg["rl_steps"]):
        model.eval()
        seqs = rl_t[torch.randint(rl_t.shape[0], (cfg["batch"],), generator=gen)].to(device)
        prefix, true_suffix = seqs[:, :PREFIX_LEN], seqs[:, PREFIX_LEN:]
        g = generate_suffix(model, prefix, suffix_len, cfg["temperature"])
        r = compute_reward(prefix, g, true_suffix, verifier, rules_t)
        baseline = float(r.mean()) if baseline is None else 0.95 * baseline + 0.05 * float(r.mean())
        adv = (r - baseline).detach()
        model.train()
        full = torch.cat([prefix, g], dim=1)
        logits, _ = model(full[:, :-1].contiguous())
        lp = F.log_softmax(logits, dim=-1).gather(2, full[:, 1:].unsqueeze(-1)).squeeze(-1)
        loss = -(adv[:, None] * lp[:, PREFIX_LEN - 1:]).mean()
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if (st + 1) % max(1, cfg["rl_steps"] // 4) == 0:
            rec = measure(model, st + 1)
            rec["sample_reward"] = float(r.mean())
            traj.append(rec)
            print(f"    [{label}] step {st+1}: reward {rec['sample_reward']:.4f}  "
                  f"drift {rec['kl_cond']:.4f} bits  grammatical {rec['valid_frac']:.4f}")
    print(f"    [{label}] {time.time() - t0:.0f}s")
    state = {k: p.detach().cpu().clone() for k, p in model.state_dict().items()}
    del model, opt
    if device == "cuda":
        torch.cuda.empty_cache()
    return {"trajectory": traj, "state": state}

## 5. The live replica

One setting of the dial, every arm, the same readouts as the full run — scaled to fit a free T4 in about
fifteen minutes. What is smaller: the model (4 layers × 128 wide instead of 6 × 192), the pretraining
budget, the rollout budget (49K per arm against the full run's 384K, and best-of-8 instead of best-of-16),
and the number of generations each readout is taken over (1000 per round, 3000 for the final
distances, against 4000 throughout).

That means **expect the shape, not the magnitudes**. Eight times fewer rollouts and a quarter as many
fine-tuning steps per round move the model a shorter way along whatever direction it is going. What should
survive the scaling-down is the ordering: drift climbing round over round under the validity verifier, both
seeds climbing together, and the two seeds ending up further from each other than either got from the
checkpoint they share. If the effect looks weak, the knob to raise first is `ei_prompts`: the drift is
driven by how many fine-tuning steps each round takes on the model's own selected samples, and that is
what the scaled-down budget cuts hardest.

Set `RUN_LIVE = False` to skip the training and read the full-scale results only.

In [ ]:
#@title Live-replica configuration (scaled down from the full run; edit freely) { display-mode: "form" }
RUN_LIVE      = True    #@param {type:"boolean"}
M_DEMO        = 4       #@param [2, 3, 4, 6] {type:"raw"}
RUN_EXACT_ARM = True    #@param {type:"boolean"}
RUN_REINFORCE = True    #@param {type:"boolean"}
SEED_A, SEED_B = 42, 43

CFG = dict(
    # model: the full run used 6 layers / 6 heads / 192 wide (2.68M params)
    n_layer=4, n_head=4, n_embd=128,
    # pretraining: the full run used min 3K / max 30K steps, patience 8 x 250
    pre_min=1500, pre_max=6000, pre_eval_every=250, patience=4, min_delta=0.003,
    # expert iteration: the full run used 6 rounds x 4000 prompts x best-of-16
    ei_rounds=6, ei_prompts=1024, ei_n=8, ei_sft_epochs=3,
    # REINFORCE: the full run used 6000 steps x batch 64 = 384K rollouts
    rl_steps=600,
    batch=64, lr=3e-4, temperature=1.0, seed=SEED_A,
)
N_TRAIN, N_VAL, N_MEASURE, N_FINAL = 40000, 1000, 1000, 3000
if SMOKE:
    CFG.update(n_layer=2, n_head=2, n_embd=32, pre_min=40, pre_max=80, pre_eval_every=20,
               patience=2, ei_rounds=2, ei_prompts=48, ei_n=4, ei_sft_epochs=1, rl_steps=8)
    N_TRAIN, N_VAL, N_MEASURE, N_FINAL = 2000, 128, 96, 128

_ei_budget = CFG["ei_rounds"] * CFG["ei_prompts"] * CFG["ei_n"]
_n_arms = 3 + int(RUN_EXACT_ARM)
print(f"live replica: m = {M_DEMO}, model {CFG['n_layer']}L/{CFG['n_head']}H/{CFG['n_embd']}D")
print(f"  {_n_arms} expert-iteration arms x {_ei_budget:,} rollouts"
      + (f", REINFORCE {CFG['rl_steps'] * CFG['batch']:,} rollouts" if RUN_REINFORCE else "")
      + f"  (full run: 384,000 per arm)")
print(f"  per-round readout: {N_MEASURE} sampled generations from held-out prefixes; "
      f"final readout {N_FINAL} (full run: {FCFG['n_eval']} throughout)")

In [ ]:
# ----------------------------------------------------------------------------
# Data, the shared measurement, and the pretrained checkpoint every arm starts from
# ----------------------------------------------------------------------------
RULES = generate_rules_invertible(V, S, L_DEPTH, M_DEMO, seed=RULE_SEED)
TABLES = build_rule_inverse(RULES)
CTX = {"rules_t": [torch.from_numpy(r).to(device) for r in RULES]}

if RUN_LIVE:
    t0 = time.time()
    CTX["train_t"] = torch.from_numpy(generate_sequences_batched(RULES, N_TRAIN, seed=SEED_A + 1)).long()
    CTX["val_t"] = torch.from_numpy(generate_sequences_batched(RULES, N_VAL, seed=SEED_A + 2)).long().to(device)
    CTX["rl_t"] = torch.from_numpy(generate_sequences_batched(RULES, N_TRAIN, seed=SEED_A + 3)).long()

    def make_measure(eval_seed, n):
        """Sample n suffixes from fixed held-out prefixes and read the dialect off them.

        The same prefixes for every arm and every round, so the only thing moving
        between measurements is the model. Per-round readouts use the smaller n
        (the trajectory needs one scalar); the distances in section 7 are taken
        from a bigger final readout, because they compare whole histograms.
        """
        prefixes = torch.from_numpy(
            generate_sequences_batched(RULES, n, seed=eval_seed)[:, :PREFIX_LEN]).long().to(device)

        def measure(model, tag):
            model.eval()
            gens = []
            for lo in range(0, prefixes.shape[0], 400):
                p = prefixes[lo:lo + 400]
                gens.append(torch.cat([p, generate_suffix(model, p, SEQ_LEN - PREFIX_LEN, 1.0)], dim=1))
            full = torch.cat(gens)
            root_valid = float(parse_valid_fractions_torch(full, CTX["rules_t"]).mean(dim=0)[-1])
            st = idiolect_stats(full.cpu().numpy(), RULES, PREFIX_LEN, TABLES)
            return {"round": tag, "kl_cond": st["overall"]["kl_cond"],
                    "valid_frac": st["overall"]["valid_frac"], "root_valid": root_valid,
                    "levels": {k: st[k] for k in st if k.startswith("L")}}
        return measure

    MEASURE = make_measure(FCFG["eval_seed"], N_MEASURE)
    MEASURE_BIG = make_measure(FCFG["eval_seed"], N_FINAL)
    MEASURE_BIG_ALT = make_measure(FCFG["eval_seed"] + 1, N_FINAL)   # independent samples

    print("pretraining the one checkpoint every arm starts from...")
    PRETRAINED, PRE_CURVE = pretrain_to_plateau(CTX, CFG)
    _m = GPT(V, SEQ_LEN, CFG["n_layer"], CFG["n_head"], CFG["n_embd"], quiet=True).to(device)
    _m.load_state_dict(PRETRAINED)
    BASE = MEASURE(_m, "pretrained")
    BASE_BIG, BASE_BIG_ALT = MEASURE_BIG(_m, "pretrained"), MEASURE_BIG_ALT(_m, "pretrained")
    del _m
    print(f"  pretrained checkpoint: drift {BASE['kl_cond']:.4f} bits "
          f"(corpus floor ~{DGP_FLOOR[M_DEMO]:.4f}), grammatical {BASE['valid_frac']:.4f}, "
          f"root validity {BASE['root_valid']:.3f}")
    print(f"  measurement noise on the dialect, same model / independent samples: "
          f"{dialect_distance(BASE_BIG['levels'], BASE_BIG_ALT['levels']):.4f}")
    print(f"  total {time.time() - t0:.0f}s")
else:
    print("RUN_LIVE is off — skipping the live replica; the full-scale sections below still run.")

In [ ]:
# ----------------------------------------------------------------------------
# The arms. All four start from the identical PRETRAINED state dict.
# ----------------------------------------------------------------------------
RUNS = {}
if RUN_LIVE:
    t0 = time.time()
    print("EI under the validity verifier, seed 42 (the main arm)")
    RUNS["parse_A"] = run_ei(PRETRAINED, "parse", SEED_A, CTX, CFG, MEASURE, "parse s42")
    print("EI under the validity verifier, seed 43 — same checkpoint, verifier, budget")
    RUNS["parse_B"] = run_ei(PRETRAINED, "parse", SEED_B, CTX, CFG, MEASURE, "parse s43")
    print("the same-seed rerun — identical config AND seed; the noise floor")
    RUNS["parse_A2"] = run_ei(PRETRAINED, "parse", SEED_A, CTX, CFG, MEASURE, "parse s42 rerun")
    if RUN_EXACT_ARM:
        print("EI under the canonical-answer verifier — the blind-spot control")
        RUNS["exact_A"] = run_ei(PRETRAINED, "exact", SEED_A, CTX, CFG, MEASURE, "exact s42")
    if RUN_REINFORCE:
        print("vanilla REINFORCE under the validity verifier — the collapse contrast")
        RUNS["reinforce"] = run_reinforce(PRETRAINED, "parse", SEED_A, CTX, CFG, MEASURE, "REINFORCE")
    # a bigger final readout of every arm: the distances compare whole histograms,
    # which need more samples than the per-round scalar does
    print("final readouts...")
    FINAL = {"pretrained": BASE_BIG}
    for k in RUNS:
        _m = GPT(V, SEQ_LEN, CFG["n_layer"], CFG["n_head"], CFG["n_embd"], quiet=True).to(device)
        _m.load_state_dict(RUNS[k]["state"])
        FINAL[k] = MEASURE_BIG(_m, "final")
        if k == "parse_A":
            FINAL["parse_A_resampled"] = MEASURE_BIG_ALT(_m, "final")
        del _m
    print(f"\nall arms done in {time.time() - t0:.0f}s")

## 6. Result 1 — each run settles on a different word for the same thing

The grammar's bottom-level rules expand a feature straight into two leaf tokens. So at that level a
`(feature, rule)` pair *is* a literal word, a feature's `m` rules are `m` words for one thing, and the
joint histogram the measurement already collects says exactly **how often each model says each word**. No
new sampling and no new training — the bits and the distances below are just this table, aggregated.

By construction the corpus says each of a feature's `m` words equally often. The question is what each run
does to that.

In [ ]:
# ----------------------------------------------------------------------------
# Reading a dictionary entry out of the per-level histograms
# ----------------------------------------------------------------------------
BOTTOM = f"L{L_DEPTH - 1}"          # the level whose rules expand straight to tokens

def conditional(levels, level=BOTTOM):
    """P(rule | feature) as a (v, m) array, from a per-level joint histogram."""
    j = np.asarray(levels[level]["joint_hist"], dtype=np.float64)
    return j / np.clip(j.sum(axis=1, keepdims=True), 1e-30, None)

def most_divergent_feature(levels_a, levels_b, level=BOTTOM):
    """The feature whose synonym distribution the two runs disagree on most."""
    d = conditional(levels_a, level) - conditional(levels_b, level)
    return int(np.argmax(np.abs(d).max(axis=1)))

def vocabulary_spread(pre, a, b, rerun, level=BOTTOM, thresh=5.0):
    """How many of the level's v*m words each run moved by more than `thresh` points."""
    p = conditional(pre, level)
    out = {}
    for name, arm in (("seed 42 vs pretrained", a), ("seed 43 vs pretrained", b),
                      ("seed 42 vs its own rerun", rerun)):
        ref = conditional(a, level) if "rerun" in name else p
        d = np.abs(conditional(arm, level) - ref).ravel() * 100
        out[name] = (int((d > thresh).sum()), int(d.size), float(d.max()))
    return out


def plot_lexicon(words, rows, rerun_delta, traj_word, traj_a, traj_b, corpus_rate, title):
    """Left: what each run did to one feature's words. Right: one word, round by round."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.4, 2.6 + 0.5 * len(words)),
                                   gridspec_kw={"width_ratios": [1.12, 1]})
    y = np.arange(len(words))[::-1].astype(float)
    h = 0.33
    xa = max(abs(np.array([v for _, vs, _ in rows for v in vs])).max(), 1.0)
    for off, (lab, vals, col) in zip((h / 2, -h / 2), rows):
        ax1.barh(y + off, vals, height=h, color=col, label=lab, zorder=3)
        for yy, vv in zip(y + off, vals):
            ax1.text(vv + np.sign(vv) * 0.03 * xa, yy, f"{vv:+.1f}", va="center",
                     ha="left" if vv >= 0 else "right", fontsize=8.5, color=PAL["muted"])
    ax1.plot(rerun_delta, y + h / 2, marker="|", ls="none", ms=13, mew=1.8,
             color=PAL["ink"], zorder=5, label="seed 42, retrained at the same seed")
    ax1.axvline(0, color=PAL["ink"], lw=1.1, zorder=2)
    ax1.set_yticks(y, [f"\u201c{w}\u201d" for w in words])
    ax1.set_xlabel("change from the pretrained model, in uses per 100")
    ax1.set_title(title, fontsize=10.5, loc="left", color=PAL["ink"])
    ax1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.30), ncol=3, fontsize=8.5)
    ax1.grid(axis="y", visible=False)
    ax1.set_xlim(-xa * 1.32, xa * 1.32)
    ax1.set_ylim(y.min() - 0.6, y.max() + 0.6)

    r = np.arange(len(traj_a))
    ax2.axhline(corpus_rate, color=PAL["gray"], ls="--", lw=1.1, zorder=2)
    ax2.annotate(f"corpus {corpus_rate:.1f}", (r[-1], corpus_rate), textcoords="offset points",
                 xytext=(-4, 5), ha="right", fontsize=8.5, color=PAL["muted"])
    for vals, col, lab in ((traj_a, PAL["seedA"], "seed 42"), (traj_b, PAL["seedB"], "seed 43")):
        ax2.plot(r, vals, "-o", color=col, lw=1.8, ms=5, label=lab, zorder=3)
        ax2.annotate(f"{vals[-1]:.1f}", (r[-1], vals[-1]), textcoords="offset points",
                     xytext=(6, 0), color=col, fontsize=9, fontweight="bold", va="center")
    ax2.set_xlabel("expert-iteration round  (0 = the shared checkpoint)")
    ax2.set_ylabel(f"uses of \u201c{traj_word}\u201d per 100")
    ax2.set_title(f"how often the model says \u201c{traj_word}\u201d", fontsize=10.5,
                  loc="left", color=PAL["ink"])
    ax2.set_xticks(r)
    ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.30), ncol=2, fontsize=8.5)
    ax2.set_xlim(-0.25, r[-1] + 0.85)
    fig.tight_layout()
    plt.show()

In [ ]:
if RUN_LIVE:
    A, B = RUNS["parse_A"]["trajectory"], RUNS["parse_B"]["trajectory"]
    feat = most_divergent_feature(FINAL["parse_A"]["levels"], FINAL["parse_B"]["levels"])
    ws = words_of(RULES, feat)
    pre_c = conditional(FINAL["pretrained"]["levels"])
    a_c, b_c, a2_c = (conditional(FINAL[k]["levels"]) for k in ("parse_A", "parse_B", "parse_A2"))
    da = [(a_c[feat, r] - pre_c[feat, r]) * 100 for r in range(M_DEMO)]
    db = [(b_c[feat, r] - pre_c[feat, r]) * 100 for r in range(M_DEMO)]
    d2 = [(a2_c[feat, r] - pre_c[feat, r]) * 100 for r in range(M_DEMO)]

    print(f"live replica, m = {M_DEMO}, feature {feat} — the words each run reaches for")
    print(f"the corpus says each of its {M_DEMO} words {100 / M_DEMO:.1f}% of the time, by construction\n")
    print(f"{'':>20} " + " ".join(f"{w:>7}" for w in ws))
    for name, cnd in (("pretrained", pre_c), ("EI seed 42", a_c),
                      ("same-seed rerun", a2_c), ("EI seed 43", b_c)):
        print(f"{name:>20} " + " ".join(f"{cnd[feat, r] * 100:>7.1f}" for r in range(M_DEMO)))

    top = int(np.argmax(a_c[feat]))
    tj_a = [conditional(x["levels"])[feat, top] * 100 for x in A]
    tj_b = [conditional(x["levels"])[feat, top] * 100 for x in B]
    plot_lexicon(ws, [("seed 42", da, PAL["seedA"]), ("seed 43", db, PAL["seedB"])], d2,
                 ws[top], tj_a, tj_b, 100 / M_DEMO,
                 f"live replica — one feature's {M_DEMO} words, m = {M_DEMO}")

    sp = vocabulary_spread(FINAL["pretrained"]["levels"], FINAL["parse_A"]["levels"],
                           FINAL["parse_B"]["levels"], FINAL["parse_A2"]["levels"], thresh=3.0)
    print(f"\nacross the whole bottom-level vocabulary ({V * M_DEMO} words), moved by >3 points:")
    for name, (n_moved, n_tot, mx) in sp.items():
        print(f"  {name:>26}: {n_moved:>3}/{n_tot}   (largest move {mx:.1f})")

### At full scale

The entry below is from the full run at `m = 3`: the feature whose synonyms the two seeds disagree on most,
its three words, and what six rounds of expert iteration did to each of them.

The pretrained model is within 0.8 points of the corpus on all three words — it speaks the corpus's dialect.
Six rounds later, **seed 42 has made “hh” its word**, taking it from 32.9 uses per 100 to 48.3 while
dropping the other two. Seed 43, from the identical checkpoint under the identical verifier with the
identical budget, went the other way on all three: it uses “hh” *less*, 27.6 per 100. On the word seed 42
committed to, the two runs end **20 points apart**.

Retraining seed 42 at the same seed lands on all three words again — the tick marks. That is the noise
floor, and it is nowhere near 20 points.

The pattern holds across the level's whole 24-word vocabulary: seed 42 moved 10 of the 24 words by more
than 5 uses per 100, seed 43 moved 8, and **only 2 are on both lists**. The same-seed rerun moved none of
them; its largest disagreement with the original run is 1.7 points. The two runs rewrote almost disjoint
parts of the dictionary.

In [ ]:
M_LEX = 3
lex_rules = generate_rules_invertible(V, S, L_DEPTH, M_LEX, seed=FCFG["rule_seed"])
lev = {k: fs(M_LEX, k)["levels"] for k in
       ("pretrained", "ei_parse", "ei_parse_rerun", "ei_parse_seed43")}
feat = most_divergent_feature(lev["ei_parse"], lev["ei_parse_seed43"])
ws = words_of(lex_rules, feat)
pre_c = conditional(lev["pretrained"])
a_c, b_c, a2_c = (conditional(lev[k]) for k in ("ei_parse", "ei_parse_seed43", "ei_parse_rerun"))

print(f"full scale, m = {M_LEX}, feature {feat} — one dictionary entry, in uses per 100")
print(f"the corpus says each of its {M_LEX} words {100 / M_LEX:.1f}% of the time, by construction\n")
print(f"{'':>20} " + " ".join(f"{w:>7}" for w in ws))
for name, lab in (("corpus", "dgp"), ("pretrained", "pretrained"), ("+ continued NTP", "pretrain_only"),
                  ("+ KL anchor", "kl_parse"), ("EI seed 42", "ei_parse"),
                  ("same-seed rerun", "ei_parse_rerun"), ("EI seed 43", "ei_parse_seed43")):
    c = conditional(fs(M_LEX, lab)["levels"])
    print(f"{name:>20} " + " ".join(f"{c[feat, r] * 100:>7.1f}" for r in range(M_LEX)))

top = int(np.argmax(a_c[feat]))
tj_a = [pre_c[feat, top] * 100] + [conditional(fs(M_LEX, f"ei_parse_r{r}")["levels"])[feat, top] * 100
                                   for r in range(1, 7)]
tj_b = [pre_c[feat, top] * 100] + [conditional(fs(M_LEX, f"ei_parse_seed43_r{r}")["levels"])[feat, top] * 100
                                   for r in range(1, 7)]
plot_lexicon(ws,
             [("seed 42", [(a_c[feat, r] - pre_c[feat, r]) * 100 for r in range(M_LEX)], PAL["seedA"]),
              ("seed 43", [(b_c[feat, r] - pre_c[feat, r]) * 100 for r in range(M_LEX)], PAL["seedB"])],
             [(a2_c[feat, r] - pre_c[feat, r]) * 100 for r in range(M_LEX)],
             ws[top], tj_a, tj_b, 100 / M_LEX,
             f"full scale — one feature's {M_LEX} words, m = {M_LEX}")

sp = vocabulary_spread(lev["pretrained"], lev["ei_parse"], lev["ei_parse_seed43"], lev["ei_parse_rerun"])
print(f"across the whole bottom-level vocabulary ({V * M_LEX} words), moved by >5 points:")
for name, (n_moved, n_tot, mx) in sp.items():
    print(f"  {name:>26}: {n_moved:>3}/{n_tot}   (largest move {mx:.1f})")
moved_a = set(np.flatnonzero((np.abs(a_c - pre_c) * 100 > 5).ravel()).tolist())
moved_b = set(np.flatnonzero((np.abs(b_c - pre_c) * 100 > 5).ravel()).tolist())
print(f"\n  words on both runs' lists: {len(moved_a & moved_b)} "
      f"(of {len(moved_a)} and {len(moved_b)})")

## 7. Result 2 — the same distance travelled, in different directions

One dictionary entry is an anecdote. The aggregate is a geometry.

Take three models — the shared pretrained checkpoint and the two runs that started from it — and measure
all three pairwise distances between their feature-conditioned synonym distributions. Because $\sqrt{JS}$
is a metric (the divergence itself is not), the three distances form a genuine triangle, and the angle at
the pretrained checkpoint follows from the law of cosines. That angle is the answer to "did they drift the
same way?":

- **0°** — the two runs found the *same* convention.
- **90°** — they share nothing; each moved in a direction the other did not.
- **Past 60°** the runs end up **further from each other than either travelled from their common
  ancestor** — the chord is longer than either arrow.

In [ ]:
def plot_triangle(ax, d_a, d_b, d_ab, floor, title, show_legend=False, label_scale=1.0, lim=None):
    """pretrained at the origin; one arrow per run; the chord between their tips."""
    ang = drift_angle(d_a, d_b, d_ab)
    scale = max(d_a, d_b) if lim is None else lim      # frame size; label offsets follow it
    th = np.radians(ang) / 2
    A = (-np.sin(th) * d_a, np.cos(th) * d_a)
    B = (np.sin(th) * d_b, np.cos(th) * d_b)
    if floor > 0:
        for P in (A, B):
            ax.add_patch(Circle(P, floor, color=PAL["gray"], alpha=0.32, lw=0, zorder=2))
    ax.plot([A[0], B[0]], [A[1], B[1]], ls="--", lw=1.3, color=PAL["muted"], zorder=3)
    ax.annotate("", xy=A, xytext=(0, 0), zorder=4,
                arrowprops=dict(arrowstyle="-|>", color=PAL["seedA"], lw=2.0, shrinkA=0, shrinkB=0))
    ax.annotate("", xy=B, xytext=(0, 0), zorder=4,
                arrowprops=dict(arrowstyle="-|>", color=PAL["seedB"], lw=2.0, shrinkA=0, shrinkB=0))
    fs_ = 9 * label_scale
    ax.text(A[0] * 0.55 - 0.055 * scale, A[1] * 0.55, f"{d_a:.3f}", color=PAL["seedA"],
            ha="right", va="center", fontsize=fs_, fontweight="bold")
    ax.text(B[0] * 0.55 + 0.055 * scale, B[1] * 0.55, f"{d_b:.3f}", color=PAL["seedB"],
            ha="left", va="center", fontsize=fs_, fontweight="bold")
    ax.text((A[0] + B[0]) / 2, max(A[1], B[1]) + floor + 0.05 * scale, f"{d_ab:.3f}",
            ha="center", va="bottom", fontsize=fs_, fontweight="bold", color=PAL["ink"])
    rad = 0.30 * min(d_a, d_b)
    ax.add_patch(Wedge((0, 0), rad, 90 - np.degrees(th), 90 + np.degrees(th), width=rad * 0.02,
                       color=PAL["muted"], zorder=3))
    ax.text(0, rad * 1.35, f"{ang:.1f}°", ha="center", fontsize=fs_ + 0.5,
            fontweight="bold", color=PAL["ink"])
    ax.plot([0], [0], "o", ms=6, color=PAL["ink"], zorder=5)
    ax.text(0, -0.045 * scale, "pretrained", ha="center", va="top",
            fontsize=fs_ - 0.5, color=PAL["muted"])
    top = max(A[1], B[1]) + floor + 0.20 * scale
    ax.set_xlim(-scale * 1.30, scale * 1.30)
    ax.set_ylim(-scale * 0.34, scale * 1.32 if lim is not None else top)
    ax.set_aspect("equal"); ax.set_axis_off()
    ax.set_title(title, fontsize=10.5, color=PAL["ink"])
    if show_legend:
        ax.plot([], [], color=PAL["seedA"], lw=2.0, label="seed 42")
        ax.plot([], [], color=PAL["seedB"], lw=2.0, label="seed 43")
        ax.plot([], [], color=PAL["muted"], lw=1.3, ls="--", label="run $\\leftrightarrow$ run")
        ax.plot([], [], color=PAL["gray"], lw=6, alpha=0.32, label="same-seed rerun: the noise floor")
        ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.05), ncol=2, fontsize=8.5)

In [ ]:
if RUN_LIVE:
    lv = {k: FINAL[k]["levels"] for k in FINAL}
    d_a = dialect_distance(lv["pretrained"], lv["parse_A"])
    d_b = dialect_distance(lv["pretrained"], lv["parse_B"])
    d_ab = dialect_distance(lv["parse_A"], lv["parse_B"])
    d_rr = dialect_distance(lv["parse_A"], lv["parse_A2"])
    d_meas = dialect_distance(lv["parse_A"], lv["parse_A_resampled"])
    floor = max(d_rr, d_meas)

    print(f"live replica, m = {M_DEMO} — dialect distances (sqrt JS), {N_FINAL} generations each")
    print(f"  pretrained -> seed 42            {d_a:.4f}")
    print(f"  pretrained -> seed 43            {d_b:.4f}")
    print(f"  seed 42 <-> seed 43              {d_ab:.4f}")
    print(f"  seed 42 <-> its same-seed rerun  {d_rr:.4f}   <- retrained at the same seed")
    print(f"  seed 42 <-> itself, resampled    {d_meas:.4f}   <- same model, independent samples")
    print(f"  angle at pretrained              {drift_angle(d_a, d_b, d_ab):.1f}°")
    print(f"  cross-seed / noise floor         {d_ab / max(floor, 1e-9):.1f}x")
    if d_rr < d_meas * 0.2:
        print("\n  note: the same-seed rerun read ~0, which means this ran somewhere with fully\n"
              "  deterministic kernels (CPU). On GPU the two runs do diverge slightly, because\n"
              "  embedding gradients accumulate nondeterministically. Either way the resampling\n"
              "  row is the floor that matters — see the check below.")
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    plot_triangle(ax, d_a, d_b, d_ab, floor, f"live replica, m = {M_DEMO}", show_legend=True)
    fig.tight_layout(); plt.show()

### At full scale

At `m = 4`: the two runs are **0.100 apart from each other**, having travelled 0.095 and 0.090 from the
checkpoint they share — the chord is longer than either arrow, at an angle of **65.3°**. The same-seed
rerun moves 0.018, so the cross-seed disagreement is **5.6× the noise floor**. The gray discs on the figure
are that floor, drawn to scale.

The same triangle exists at every `m`, and it is the same shape. The *amount* of drift replicates closely
between the seeds at every setting — 0.065 vs 0.059 at `m = 2`, 0.124 vs 0.115 at `m = 6`. The *direction*
does not: 74.8°, 78.5°, 65.3°, 45.3°, against a noise floor 4–6.6× smaller. At `m ≤ 4` the two runs end
further from each other than from their common ancestor.

**Reproducible in amount, arbitrary in direction.** That is the signature of an idiolect rather than a
correction: a mechanism converging on a genuinely better convention would have both runs find approximately
the same one.

The angle does narrow toward `m = 6`, and that narrowing is real (bootstrap ±2°, against a pure-noise null
of 60°). A [child experiment](https://github.com/jagilley/abstraction/tree/main/experiments/rhm/rl_dimensionality/idiolect/direction)
resolves it: it is a level mixture. At the shallow levels, where the verifier filters hard because the model
often fails, both seeds are pulled toward the synonyms the model already executes most reliably — the shared
part. At the deep levels, where the model is already fluent and the verifier rarely rejects anything, the
drift is near-orthogonal — the private part. A selection-free control (random reward) removes the shared
component at every level; a fresh *pretraining* seed reproduces it, because which synonyms are hard is a
property of the grammar, not of the run.

In [ ]:
def full_triangle(m, arm="parse"):
    """(pretrained->A, pretrained->B, A<->B, same-seed floor) at multiplicity m."""
    lv = {k: fs(m, k)["levels"] for k in
          ("pretrained", f"ei_{arm}", f"ei_{arm}_rerun", f"ei_{arm}_seed43")}
    return (dialect_distance(lv["pretrained"], lv[f"ei_{arm}"]),
            dialect_distance(lv["pretrained"], lv[f"ei_{arm}_seed43"]),
            dialect_distance(lv[f"ei_{arm}"], lv[f"ei_{arm}_seed43"]),
            dialect_distance(lv[f"ei_{arm}"], lv[f"ei_{arm}_rerun"]))

fig, ax = plt.subplots(figsize=(6.0, 4.0))
plot_triangle(ax, *full_triangle(4), "full scale, $m = 4$: the whole triangle", show_legend=True)
fig.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(13.6, 4.1))
FAN = {m: full_triangle(m) for m in (2, 3, 4, 6)}
span = max(max(t[0], t[1]) for t in FAN.values())      # one frame for all four
for ax, m in zip(axes, (2, 3, 4, 6)):
    d_a, d_b, d_ab, d_rr = FAN[m]
    plot_triangle(ax, d_a, d_b, d_ab, d_rr, f"$m = {m}$", label_scale=0.92, lim=span)
    ax.text(0, -0.245 * span, f"cross-seed = {d_ab / d_rr:.1f}× same-seed",
            ha="center", va="top", fontsize=8.5, color=PAL["muted"])
axes[0].plot([], [], color=PAL["seedA"], lw=2.0, label="seed 42")
axes[0].plot([], [], color=PAL["seedB"], lw=2.0, label="seed 43")
axes[0].plot([], [], color=PAL["muted"], lw=1.3, ls="--", label="run $\\leftrightarrow$ run")
axes[0].plot([], [], color=PAL["gray"], lw=6, alpha=0.32, label="same-seed rerun")
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=4, fontsize=9,
           bbox_to_anchor=(0.5, -0.03))
fig.suptitle("Equal distance travelled, private directions — the same triangle at every $m$",
             fontsize=11, color=PAL["ink"], y=1.02)
fig.tight_layout(); plt.show()

print(f"{'m':>3}  {'pret->s42':>10}  {'pret->s43':>10}  {'s42<->s43':>10}  "
      f"{'same-seed':>10}  {'angle':>7}  {'ratio':>6}")
for m in (2, 3, 4, 6):
    d_a, d_b, d_ab, d_rr = full_triangle(m)
    print(f"{m:>3}  {d_a:>10.4f}  {d_b:>10.4f}  {d_ab:>10.4f}  {d_rr:>10.4f}  "
          f"{drift_angle(d_a, d_b, d_ab):>6.1f}°  {d_ab / d_rr:>5.1f}x")

### What is actually in that noise floor?

The "same-seed rerun" arm is two things at once: a retrain whose only difference is float
nondeterminism, *and* a fresh set of 4000 sampled generations. So part of the floor is the retrain and
part is just sampling noise on a histogram. The cell below separates them, with no training and no
checkpoints: draw two independent corpora of 4000 sequences from the grammar itself and measure the
distance between them. Anything above zero there is pure sampling.

It turns out the same-seed rerun floor is **almost entirely sampling noise** — the two columns agree to
within a couple of thousandths at every `m`. That is the conservative direction to be wrong in. The floor
the cross-seed distances are quoted against is, if anything, an over-estimate of how much two retrained
runs really differ, so the 4–6.6× ratios are if anything understated. (And the same noise contributes
almost nothing to the numerator: at `m = 4` it is 3% of the cross-seed divergence.)

In [ ]:
n_pair = 300 if SMOKE else FCFG["n_eval"]
pairs = [(101, 202)] if SMOKE else [(101, 202), (303, 404), (505, 606)]
print(f"pure sampling noise on the dialect distance, {n_pair} generations per side, "
      f"{len(pairs)} independent pairs\n")
print(f"  {'m':>3}  {'two corpora, resampled':>24}  {'same-seed rerun (full scale)':>30}")
for m in (2, 3, 4, 6):
    rules = generate_rules_invertible(V, S, L_DEPTH, m, seed=RULE_SEED)
    ds = []
    for sa, sb in pairs:
        lv = [{k: v for k, v in idiolect_stats(
                  generate_sequences_batched(rules, n_pair, seed=sd), rules, PREFIX_LEN).items()
               if k.startswith("L")} for sd in (sa, sb)]
        ds.append(dialect_distance(*lv))
    print(f"  {m:>3}  {np.mean(ds):>24.4f}  {full_triangle(m)[3]:>30.4f}")

## 8. Result 3 — the drift arrives packaged with improvement, not damage

Drift on its own could just be damage: a model getting worse in every way, including this one. The
round-by-round picture rules that out. Three quantities, measured on the same generations after every
round:

- **drift** from the corpus's synonym habits, in bits — the thing the verifier cannot see;
- **root validity** — the fraction of generations that parse all the way to a root. This is essentially
  the reward;
- **grammaticality** — the fraction of the model's own nodes that parse at all.

In [ ]:
def plot_rounds(rounds, series, base, floor=None, suptitle=None):
    """Three panels sharing the round axis; one measure each, one line per seed.

    `series` entries are (label, values, color, linestyle). Linestyle carries the
    verifier (solid = validity, dashed = canonical-answer); color carries the seed,
    so a seed keeps its colour across every figure in the notebook.
    """
    panels = [("kl_cond", "drift from the corpus (bits)", "{:.4f}", floor),
              ("root_valid", "root validity \u2014 the reward", "{:.3f}", None),
              ("valid_frac", "grammaticality of the model's nodes", "{:.4f}", None)]
    fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.5))
    for ax, (key, ylab, fmt, ref) in zip(axes, panels):
        ax.axhline(base[key], color=PAL["gray"], ls="--", lw=1.2, zorder=2)
        ax.annotate("pretrained " + fmt.format(base[key]), (rounds[-1], base[key]),
                    textcoords="offset points", xytext=(-2, 4), ha="right",
                    fontsize=8.5, color=PAL["muted"])
        if ref is not None:
            ax.axhline(ref, color=PAL["muted"], ls=":", lw=1.0, zorder=2)
            ax.annotate(f"corpus floor {ref:.4f}", (rounds[-1], ref), textcoords="offset points",
                        xytext=(-2, -11), ha="right", fontsize=8.5, color=PAL["muted"])
        for lab, vals, col, ls in series:
            ax.plot(rounds, vals[key], ls, marker="o", color=col, lw=1.9, ms=5, label=lab, zorder=3)
            ax.annotate(fmt.format(vals[key][-1]), (rounds[-1], vals[key][-1]),
                        textcoords="offset points", xytext=(6, 0), color=col, fontsize=9,
                        fontweight="bold", va="center")
        ax.set_xlabel("expert-iteration round")
        ax.set_ylabel(ylab)
        ax.set_xticks(rounds)
        ax.set_xlim(rounds[0] - 0.2, rounds[-1] + 1.0)
        if ref is not None:
            lo, hi = ax.get_ylim()
            ax.set_ylim(lo - 0.10 * (hi - lo), hi)
    axes[0].legend(loc="upper left", fontsize=8.5)
    if suptitle:
        fig.suptitle(suptitle, fontsize=11, color=PAL["ink"], y=1.03)
    fig.tight_layout()
    plt.show()


def series_from(traj, keys=("kl_cond", "root_valid", "valid_frac")):
    return {k: [t[k] for t in traj] for k in keys}

In [ ]:
if RUN_LIVE:
    rounds = list(range(CFG["ei_rounds"] + 1))
    plot_rounds(rounds,
                [("seed 42", series_from(RUNS["parse_A"]["trajectory"]), PAL["seedA"], "-"),
                 ("seed 43", series_from(RUNS["parse_B"]["trajectory"]), PAL["seedB"], "-")],
                BASE, floor=DGP_FLOOR[M_DEMO],
                suptitle=f"live replica \u2014 the validity verifier (blind to word choice), m = {M_DEMO}")
    if RUN_EXACT_ARM:
        plot_rounds(rounds,
                    [("seed 42, canonical-answer verifier",
                      series_from(RUNS["exact_A"]["trajectory"]), PAL["seedA"], "--")],
                    BASE, floor=DGP_FLOOR[M_DEMO],
                    suptitle=f"live replica \u2014 the same loop under the verifier that CAN see "
                             f"word choice, m = {M_DEMO}")

### At full scale

At `m = 6`, over six rounds of expert iteration under the validity verifier, in both seeds:

| round | drift (bits), s42 / s43 | root validity | grammaticality |
|---|---|---|---|
| 1 | 0.0134 / 0.0139 | 0.167 / 0.153 | 0.921 / 0.919 |
| 3 | 0.0395 / 0.0379 | 0.211 / 0.198 | 0.935 / 0.933 |
| 6 | **0.0792 / 0.0721** | **0.306 / 0.282** | **0.952 / 0.947** |

All three climb together, monotonically, in both seeds. The final model is **more grammatical than the
pretrained model it started from** (0.952 against 0.914), has roughly **doubled its pass rate** (0.141 →
0.306), and sits **13× further from the corpus's synonym habits** (0.0058 → 0.0783 bits). Nothing about
the output is degraded. It is just progressively less the corpus's dialect.

The canonical-answer verifier is the contrast that shows the blind spot is doing the work. Run the same
six rounds under a verifier that *can* see word choice — where using a synonym counts as wrong — and drift
still climbs (0.0085 → 0.0262), but root validity **falls** (0.130 → 0.111) and grammaticality **falls**
(0.909 → 0.899). Under the invariant verifier, drift comes packaged with gains. Under the canonical one,
it is just damage.

This is the parent experiment's central dissociation restated on a new axis. There, validity climbed while
the model's knowledge of the grammar's deep rules stayed pinned at uniform. Here, validity climbs while the
model's way of *saying* things moves steadily away from everyone else's. In both cases the verifier's
number improves while something it cannot see comes loose.

In [ ]:
def full_rounds(m, arm, seed_tag=""):
    suffix = f"_{seed_tag}" if seed_tag else ""
    recs = [fs(m, f"ei_{arm}{suffix}_r{r}") for r in range(1, 7)]
    return {"kl_cond": [r["overall"]["kl_cond"] for r in recs],
            "root_valid": [r["root_valid"] for r in recs],
            "valid_frac": [r["overall"]["valid_frac"] for r in recs]}

M_ROUNDS = 6
base6 = {"kl_cond": fs(M_ROUNDS, "pretrained")["overall"]["kl_cond"],
         "root_valid": fs(M_ROUNDS, "pretrained")["root_valid"],
         "valid_frac": fs(M_ROUNDS, "pretrained")["overall"]["valid_frac"]}

plot_rounds(list(range(1, 7)),
            [("seed 42", full_rounds(M_ROUNDS, "parse"), PAL["seedA"], "-"),
             ("seed 43", full_rounds(M_ROUNDS, "parse", "seed43"), PAL["seedB"], "-")],
            base6, floor=fs(M_ROUNDS, "dgp")["overall"]["kl_cond"],
            suptitle="Full scale, $m=6$ \u2014 the validity verifier, which cannot see word choice: "
                     "everything climbs together")

plot_rounds(list(range(1, 7)),
            [("seed 42", full_rounds(M_ROUNDS, "exact"), PAL["seedA"], "--"),
             ("seed 43", full_rounds(M_ROUNDS, "exact", "seed43"), PAL["seedB"], "--")],
            base6, floor=fs(M_ROUNDS, "dgp")["overall"]["kl_cond"],
            suptitle="Full scale, $m=6$ \u2014 the canonical-answer verifier, which can: "
                     "the drift is still there, the gains are not")

## 9. How it scales with synonymy — and what collapse looks like, for contrast

The blind spot is a property of the *pair* (verifier, domain), so it should have a signature in `m`. It
does, and the two verifiers run in opposite directions:

- Under the **validity** verifier the drift **rises** with synonymy: 0.017 → 0.078 bits across `m = 2 → 6`.
  More ways to say the same thing means more room to be idiosyncratic in a way the grader never sees.
- Under the **canonical-answer** verifier it **falls** over the same range: 0.055 → 0.024. This follows
  from the parent's bound: the canonical verifier's prefix-conditional headroom collapses with `m`, so
  best-of-16 selection becomes near-random, and fine-tuning on near-randomly selected samples has no
  systematic direction to push.

The curves cross around `m = 3–4`. It is the same verifier × dimensionality reversal the parent found for
reward *density*, appearing here on a completely independent measurement.

Two controls sit underneath. **`m = 1` reads exactly 0.0000 in every condition** — with one rule per
feature there is no synonym to drift into. And the pretrained model and a step-matched continued-NTP
control both sit within ~5× the corpus floor, so the metric is not picking up generic training noise.

**A KL anchor nearly eliminates the effect** (0.0125 against expert iteration's 0.0783 at `m = 6`), which
prices it: a KL penalty to the pretrained policy is precisely a penalty on distributional deviation, word
choice included.

**And this is not entropy collapse.** Vanilla REINFORCE with no anchor saturates at exactly $\log_2 m$ at
every `m` — one synonym for everything, total commitment — while destroying grammaticality. That is a
known pathology and it is uninformative about the question here. Expert iteration drifts at about **3% of
saturation** while ending up *more* grammatical than it started. The two are different phenomena that
happen to move the same number.

In [ ]:
SWEEP = [("REINFORCE, no anchor", "reinforce_parse", PAL["collapse"], "-"),
         ("expert iteration, canonical-answer verifier", "ei_exact", PAL["exact"], "-"),
         ("expert iteration, validity verifier", "ei_parse", PAL["seedA"], "-"),
         ("+ KL anchor (validity verifier)", "kl_parse", PAL["anchor"], "-"),
         ("pretrained", "pretrained", PAL["gray"], "-"),
         ("corpus floor (finite-sample)", "dgp", PAL["muted"], ":")]

fig, ax = plt.subplots(figsize=(9.0, 4.6))
ms = [2, 3, 4, 6]
ax.plot(ms, [np.log2(m) for m in ms], ls="--", lw=1.1, color=PAL["ink"], alpha=0.45, zorder=2)
ax.annotate("$\\log_2 m$ \u2014 always the same synonym", (3.4, np.log2(3.4)),
            textcoords="offset points", xytext=(0, 11), ha="center", va="bottom",
            fontsize=8.5, color=PAL["muted"])
for lab, key, col, ls in SWEEP:
    y = [fs(m, key)["overall"]["kl_cond"] for m in ms]
    ax.plot(ms, y, ls, marker="o", ms=5, lw=1.9, color=col, label=lab, zorder=3)
    ax.annotate(f"  {lab}", (ms[-1], y[-1]), textcoords="offset points", xytext=(6, 0),
                fontsize=8.5, color=col, va="center")
ax.set_yscale("log")
ax.set_xticks(ms)
ax.set_xlabel("$m$ — how many ways the grammar has of saying the same thing")
ax.set_ylabel("drift from the corpus's synonym habits (bits)")
ax.set_title("Under the verifier that cannot see word choice, drift grows with synonymy;\n"
             "under the one that can, it shrinks", fontsize=11, loc="left", color=PAL["ink"])
ax.set_xlim(1.85, 10.4)
fig.tight_layout(); plt.show()

print("drift, kl_cond in bits (sampled decoding, 4000 generations per checkpoint)\n")
print(f"{'mechanism':>44}  " + "  ".join(f"{'m=' + str(m):>7}" for m in MS_FULL))
for lab, key in [("corpus floor (finite-sample)", "dgp"), ("pretrained", "pretrained"),
                 ("+ continued next-token prediction", "pretrain_only"),
                 ("+ KL anchor, canonical-answer", "kl_exact"), ("+ KL anchor, validity", "kl_parse"),
                 ("REINFORCE, canonical-answer", "reinforce_exact"),
                 ("REINFORCE, validity", "reinforce_parse"),
                 ("expert iteration, canonical-answer", "ei_exact"),
                 ("expert iteration, validity", "ei_parse"),
                 ("expert iteration, validity — seed 43", "ei_parse_seed43")]:
    print(f"{lab:>44}  " + "  ".join(f"{fs(m, key)['overall']['kl_cond']:>7.4f}" for m in MS_FULL))
print(f"{'maximum possible = log2 m':>44}  " + "  ".join(f"{np.log2(m):>7.4f}" for m in MS_FULL))

In [ ]:
if RUN_LIVE and RUN_REINFORCE:
    rf = RUNS["reinforce"]["trajectory"]
    ei = RUNS["parse_A"]["trajectory"]
    print(f"live replica, m = {M_DEMO} — the collapse contrast (maximum drift = "
          f"log2 {M_DEMO} = {np.log2(M_DEMO):.4f} bits)\n")
    print(f"{'':>26} {'drift (bits)':>13} {'% of saturation':>17} {'grammaticality':>15}")
    for lab, rec in (("pretrained", ei[0]), ("expert iteration", ei[-1]),
                     (f"REINFORCE, {CFG['rl_steps']} steps", rf[-1])):
        print(f"{lab:>26} {rec['kl_cond']:>13.4f} {100 * rec['kl_cond'] / np.log2(M_DEMO):>16.1f}% "
              f"{rec['valid_frac']:>15.4f}")
    print("\nAt full scale REINFORCE reaches exactly log2(m) — total commitment to one synonym — while\n"
          "grammaticality falls to the parse floor. Here it has only had "
          f"{CFG['rl_steps']} steps, so read the direction, not the endpoint.")

## 10. What it means

Pretraining is the only force holding word choice to shared convention, because calibration to the
corpus's surface forms *is* its objective. A reward blind to word choice does not push the model toward
idiosyncrasy — it simply stops holding the coordinate in place, and the coordinate then moves under
selection noise, in a direction set by the seed.

In language terms: **a model gets better at everything the grader can see while drifting into a dialect of
one.**

### Findings

1. **A verifier invariant to surface choice lets surface choice drift, increasingly so as synonymy grows.**
   Expert iteration under the validity verifier: 0.017 → 0.078 bits across `m = 2 → 6`, while a
   canonical-answer verifier's drift shrinks over the same range (0.055 → 0.024) — the parent's verifier ×
   dimensionality reversal, on an independent measurement.
2. **The drift co-occurs with improvement in everything the verifier can see.** At `m = 6`, six rounds
   raise root validity, raise grammaticality above the pretrained model's, and raise drift 13× —
   monotonically, in both seeds. Under the canonical-answer verifier the same drift co-occurs with
   *falling* validity, which separates drift that buys reward from drift that is damage.
3. **The direction has a private part and a shared part.** Independent seeds from the identical checkpoint
   drift comparably far and 45–79° apart, 4–6.6× the same-seed noise floor; at `m ≤ 4` they end further
   from each other than from their common ancestor. The
   [child experiment](https://github.com/jagilley/abstraction/tree/main/experiments/rhm/rl_dimensionality/idiolect/direction)
   traces the shared part to the verifier's selection on grammaticality leaking onto the synonym coordinate
   through the model's uneven competence at synonyms; the private part is a random walk where the model is
   already fluent.
4. **A KL anchor nearly eliminates it** (0.0125 against 0.0783 at `m = 6`) — the same mechanism the parent
   found preserves the pretrained basis also preserves shared vocabulary.

### What this does not show

RHM synonyms are **interchangeable by construction**: there is no meaning difference between two rules at
all. So this establishes that the mechanism exists, is measurable against exact ground truth, scales with
synonymy as the framework predicts, and produces arbitrary rather than shared conventions.

It does **not** establish that this mechanism accounts for the term-invention behaviour seen in frontier
models, where an invented term plausibly *does* compress something the model is tracking. Competing
explanations are untouched here: reward models with a stylistic preference for confident jargon (a
reward-*hacking* story, where the verifier rewards neologism rather than being blind to it), within-context
self-conditioning at inference with no training involved, and training on model-generated data in general.

One substrate, one model size (2.68M parameters), one rule seed, and — for everything except the
expert-iteration arms — one training seed. The expert-iteration results are the ones carrying the claims,
and they are replicated across two seeds plus a same-seed rerun.

Two scope notes on the metric itself. Conditions with lower grammaticality contribute fewer valid nodes and
so carry slightly more finite-sample bias; this matters only for the collapsed REINFORCE arms, where the
metric is saturated anyway. And **greedy** decoding numbers (in the results JSON, not shown here) are
near-saturated for *every* model including the pretrained one — argmax decoding is deterministic, so it
necessarily commits to one synonym. They do not measure what the sampled numbers measure.

### The natural next probes

A **`kl_coef` sweep**, since the anchor suppresses the effect almost entirely and a dose–response curve
would price shared-vocabulary preservation against reward gain. And, for the real phenomenon rather than
the mechanism, a **coin-then-define test** on frontier models: elicit a coined term, then ask for a cold
definition in independent fresh contexts, against real technical terms of matched corpus rarity, testing
whether coined terms are stable in-context and unstable out of it.

### Reproducing the full-scale run

The experiment runs on [Modal](https://modal.com). From `experiments/` in
[the repo](https://github.com/jagilley/research):

```bash
# self-tests: rule recovery vs generation traces, validity vs the set parse,
# metric calibration at both endpoints (CPU, ~1 min)
modal run -m rhm.rl_dimensionality.idiolect.idiolect_drift::self_test

# the seed-43 privacy control — the only new training in this experiment;
# everything else reuses the parent's checkpoints (five detached L4 jobs)
for M in 1 2 3 4 6; do
  modal run --detach -m rhm.rl_dimensionality.rl_dim_ablation::run_setting --m $M \
    --only-ei --run-tag ei_s43 --seed 43 --save-round-ckpts \
    --pretrained-path rl_dimensionality/v8_s2_L6_m${M}_both_seed42/pretrained.pt
done

# the sweep: inference only over every checkpoint (~70 min on one L4)
modal run --detach -m rhm.rl_dimensionality.idiolect.idiolect_drift::analyze \
    --n-eval 4000 --out-tag seed43

# the tables in this notebook (results JSON is committed alongside the README)
cd rhm/rl_dimensionality/idiolect && python3 aggregate_idiolect.py
python3 -m rhm.rl_dimensionality.idiolect.lexicon --m 3   # the dictionary entry
```

Writeup: [`experiments/rhm/rl_dimensionality/idiolect/README.md`](https://github.com/jagilley/abstraction/blob/main/experiments/rhm/rl_dimensionality/idiolect/README.md).
Parent: [`../README.md`](https://github.com/jagilley/abstraction/blob/main/experiments/rhm/rl_dimensionality/README.md).
Slides: `papers/rl_idiolect_slides.tex`.